# 🧬 Aircheck Workshop: Using Machine Learning to Find Hits

**AIRCHECK Workshop 2026** · Training, evaluating and screening small molecules with models
built on chemical fingerprints.

This notebook walks the whole path: load the DEL screening data, train a LightGBM model on
molecular fingerprints, measure what it is really worth, screen a compound library, then filter and
cluster the nominees down to a shortlist worth taking to the bench.

> **What you need to submit.** Work through **Sections 1 to 8**, then run **Section 9 at the
> very end** — it writes your top 200 compounds to a CSV, and that file is your entry. The
> optional extra steps sitting between them change nothing about what you hand in.
>
> **Hackathon rules, timing, scoring and ideas to try** are in the final section of this
> notebook. Worth reading before you start the clock.

---

# 📦 Section 1 · Install and Import Dependencies

In this section we install the packages the workshop needs for chemical data processing
and machine learning.

In [ ]:
# This notebook runs both in Google Colab and from a local clone of the repository.
# In Colab it clones the repository so the workshop data is available; locally it
# simply locates the repository root. Either way the paths below are the same.
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/ShagReza/Aircheck-Workshop-2026.git"
REPO_NAME = "Aircheck-Workshop-2026"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not Path(REPO_NAME).exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    REPO_ROOT = Path(REPO_NAME).resolve()
else:
    REPO_ROOT = Path.cwd().resolve()
    while not (REPO_ROOT / "requirements.txt").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print("Running in Colab" if IN_COLAB else "Running locally")
print(f"Repository root: {REPO_ROOT}")
print(f"Data files:      {sorted(p.name for p in DATA_DIR.glob('*.parquet'))}")

In [ ]:
# Install every package the workshop needs, as listed in the repository's requirements.txt.
# In Colab this installs into the runtime. Locally we assume you already created a virtual
# environment with `python -m pip install -r requirements.txt`, so nothing is installed here.
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_ROOT / "requirements.txt")], check=True)
    print("Requirements installed.")
else:
    print("Local run - install requirements with:")
    print(f"    python -m pip install -r {REPO_ROOT / 'requirements.txt'}")

In [ ]:
# Import libraries:
import os
import sys

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from rdkit.Chem import MolFromSmiles
from rdkit.Chem import AllChem

# Make the repository's src/ package importable, then load the workshop helpers.
# REPO_ROOT was worked out in the bootstrap cell, so this behaves identically in
# Colab (where the repo was cloned) and locally.
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.metrics import hits_at_k, enrichment_at_k, screening_table
from src.plots import (plot_cv_metrics, plot_score_distribution, plot_pr_and_roc,
                       plot_enrichment_curve, plot_threshold_tradeoff,
                       plot_molecule_grid)

# The K values we report everywhere: how many actives sit in the top K compounds.
KS = (20, 50, 100, 200, 500)

print("Workshop helpers loaded from", REPO_ROOT / "src")


---

# 📂 Section 2 · Load the Workshop Data

## First: what do "train", "validation" and "test" mean?

These three words get used loosely, and mixing them up is the most common way a model
looks better than it is. They describe **roles that data plays**, not properties of the
files themselves — the same file could play a different role in a different project.

| Role | What it is for | Does the model fit on it? | May we use it to make choices? |
|---|---|---|---|
| **Training set** | the model learns its parameters from this | yes | yes |
| **Validation set** | measuring performance *while we are still deciding things* | no | yes — that is its job |
| **Test set** | one final, unbiased estimate at the very end | no | **no** — look once, change nothing after |

The single rule underneath all three: **a dataset can only give you an unbiased answer once.** The
moment you use a set to make a decision — which fingerprint, which threshold, which model —
it has taught you something, and it can no longer give you an unbiased estimate. That is why
the test set stays sealed until the end.

## How that maps onto this workshop

| File | Compounds | What it is | Role in this notebook |
|---|---|---|---|
| `sample-train.parquet` | 4,000 | DEL screen against WDR91, balanced 50/50 on `LABEL` | **training data** — Section 5 splits it into train and validation folds |
| `sample-test.parquet` | 5,000 | labelled, carries `SMILES`, only **9** actives | **test data** — opened in Section 7, and scoring it in Section 8 *is* the virtual screen |
| `sample-screen.parquet` | 5,000 | carries `SMILES`, no labels | not used in this run |

So the whole notebook is only two files:

1. **`df_train`** — we fit on it and we validate on it. In Section 5 we cut it into five
   parts: each round, four parts train the model and the fifth scores it. The held-out part
   is a **validation fold**. Nothing here is called a test set.
2. **`df_test`** — untouched until Section 7. By then every choice has been made, so the
   number it gives us is unbiased. Scoring it is also exactly what a virtual screen
   *is*, which is why this one file is both our test set and our screening library.

> **Why the test set and the screening library are the same file here**
>
> `sample-test.parquet` is the only file carrying **both** `SMILES` and `LABEL`. Using it
> as the screening library lets us do something a real screen cannot: check afterwards
> whether the compounds the model nominated were the ones that were actually active. Point
> `df_test` at `sample-screen.parquet` when you want the genuine unlabelled experience —
> then you get predictions with no way to score them, which is real life.

> **Watch the class balance change between the two.** We train on a set that is 50% active.
> The test set has **9 actives in 5,000 compounds (0.18%)**, which is what screening
> actually looks like. Keep it in mind when you read the metrics: a model that predicts
> "inactive" for everything scores 99.8% accuracy on the test set and is useless.

`df_train` is already balanced, so unlike previous years there is no downsampling step here.

In [ ]:
# Read the datasets into Pandas DataFrames. Only two files, one role each:
#
#   df_train -> TRAINING data. Section 5 splits this into train and validation folds.
#   df_test  -> TEST data. Left alone until Section 6, where scoring it is the screen.
df_train = pd.read_parquet(DATA_DIR / "sample-train.parquet")   # balanced 50/50, no SMILES
df_test = pd.read_parquet(DATA_DIR / "sample-test.parquet")   # 9 actives in 5,000, has SMILES

for name, frame, role in [
        ("df_train", df_train, "training data (split into train/validation folds)"),
        ("df_test", df_test, "test data (also the screening library)")]:
    actives = int(frame["LABEL"].sum())
    print(f"{name:<9} {frame.shape[0]:>5} rows x {frame.shape[1]:>2} cols   "
          f"actives: {actives:>4} ({actives / len(frame):>6.2%})   {role}")

## Training set

In [ ]:
# Display first few rows of the train dataset
df_train.head()

In [ ]:
# Get the list of column names from the DataFrame and print them
column_names_list = df_train.columns.tolist()
print(column_names_list)

## Test set (also our screening library)

In [ ]:
# Display first few rows of the test dataset
df_test.head()

In [ ]:
# Get the list of column names from the DataFrame and print them
column_names_list = df_test.columns.tolist()
print(column_names_list)

---

# 🧬 Section 3 · Building the Feature Matrix

A model cannot read a molecule. It needs a fixed-length row of numbers, and that is exactly
what a **molecular fingerprint** is: each position records whether — or how often — a
particular substructure appears in the molecule.

The files already carry nine fingerprints, computed for you:

| Fingerprint | Bits | What it encodes |
|---|---|---|
| `ECFP4`, `ECFP6` | 2048 | circular substructures around each atom, radius 2 and 3 |
| `FCFP4`, `FCFP6` | 2048 | the same idea, but atoms grouped by chemical *function* |
| `MACCS` | 167 | 167 hand-written yes/no structural questions |
| `RDK` | 2048 | paths through the molecular graph |
| `AVALON` | 2048 | a mixed set of substructure and path features |
| `ATOMPAIR` | 2048 | pairs of atoms and the distance between them |
| `TOPTOR` | 2048 | torsions: short four-atom fragments |

Each entry in those columns is already a NumPy array of counts, so building a feature matrix
is just stacking them into rows. We default to **ECFP4**, the most widely used of the set.

In [ ]:
# Function to turn a fingerprint column into a feature matrix
def process_data(X, column_name):
    """Stack a fingerprint column into a 2D array of shape (n_molecules, n_bits).

    The fingerprints are stored as arrays of counts, so this is just a stack - there is no
    string to parse. We cast to float32 because that is what LightGBM expects.
    (src/features.py holds the same logic, plus the fusion helpers used further down.)
    """
    return np.stack(X[column_name].to_numpy()).astype(np.float32)


# List of available fingerprint column names
fingerprint_columns = ['ECFP4', 'ECFP6', 'FCFP4', 'FCFP6', 'MACCS', 'RDK', 'AVALON', 'ATOMPAIR', 'TOPTOR']
selected_fps = 'ECFP4'  # Replace with desired fingerprints

# Feature matrices, one per role.
TrainData = process_data(df_train, selected_fps)
TestData = process_data(df_test, selected_fps)

# Labels come from the 'LABEL' column
TrainLabel = df_train['LABEL']
TestLabel = df_test['LABEL']

print(f"Fingerprint in use: {selected_fps}")
print(f"  TrainData {str(TrainData.shape):>14}   training data, to be split into folds")
print(f"  TestData  {str(TestData.shape):>14}   test data, not used until Section 6")
print()
print(f"  labels  train: {int(TrainLabel.sum()):>5} active / {int((1 - TrainLabel).sum()):>5} inactive")
print(f"           test: {int(TestLabel.sum()):>5} active / {int((1 - TestLabel).sum()):>5} inactive")

# What does a fingerprint matrix actually look like?
print()
print(f"First 5 molecules, first 12 of {TrainData.shape[1]} bits:")
display(pd.DataFrame(TrainData[:5, :12],
                     columns=[f"bit_{i}" for i in range(12)]).astype(int))

print(f"Only {(TrainData > 0).mean():.1%} of all entries are non-zero - a fingerprint is a "
      f"sparse\ndescription, mostly recording which substructures are ABSENT.")

## Optional · Fusing several fingerprints

Each fingerprint looks at the molecule differently. `ECFP4` sees circular substructures,
`MACCS` asks 167 hand-written structural questions, `RDK` walks paths through the graph.
Laying two or three side by side gives the model more angles on the same compound, and
sometimes that helps.

**Whatever you fuse, you must fuse for training and test alike.** A model trained on
`ECFP4 + MACCS` expects 2,215 numbers per molecule, in that order, forever. Hand it a
plain `ECFP4` row at prediction time and it will read MACCS questions off the end of the
circular bits and answer confidently with nonsense. So the cell below builds both.

> **Optional, and it costs you time.** Fusing makes the matrix wider, and training time
> and memory grow with the width. `ECFP4 + MACCS + RDK` is 4,263 features against ECFP4's
> 2,048 — roughly twice the work per tree, so cross-validation and the ensemble section
> will take noticeably longer. Measure the gain before you pay for it.

Nothing here touches `TrainData` or `TestData`; the results go into new names. The comment
at the end of the cell shows how to adopt a fusion if you decide it is worth it.

In [ ]:
# OPTIONAL - nothing here changes TrainData, TestData or the rest of the notebook.
from src.features import fuse_fingerprints, fusion_summary

# Each entry is one candidate feature set: a single fingerprint, or several to be
# joined side by side. "Fusing" just means concatenating them into one wider row.
fingerprint_combinations = [
    ['ECFP4'],                        # what the notebook uses by default
    ['ECFP4', 'MACCS'],               # two fingerprints joined
    ['ECFP4', 'MACCS', 'RDK'],        # three fingerprints joined
]

print("How wide does each combination get?")
print(fusion_summary(df_train, fingerprint_combinations).to_string(index=False))

# Build the fused matrices under NEW names - and build them for BOTH roles, because
# the model must see the same features at training time and at prediction time.
fusion_2 = ['ECFP4', 'MACCS']
fusion_3 = ['ECFP4', 'MACCS', 'RDK']

TrainData_fused2 = fuse_fingerprints(df_train, fusion_2)
TestData_fused2 = fuse_fingerprints(df_test, fusion_2)

TrainData_fused3 = fuse_fingerprints(df_train, fusion_3)
TestData_fused3 = fuse_fingerprints(df_test, fusion_3)

print()
print(f"{'feature set':<22} {'train':>16} {'test':>16}")
print(f"{'-' * 22} {'-' * 16} {'-' * 16}")
print(f"{'ECFP4 (in use)':<22} {str(TrainData.shape):>16} {str(TestData.shape):>16}")
print(f"{' + '.join(fusion_2):<22} {str(TrainData_fused2.shape):>16} {str(TestData_fused2.shape):>16}")
print(f"{' + '.join(fusion_3):<22} {str(TrainData_fused3.shape):>16} {str(TestData_fused3.shape):>16}")

print(f"\nMain data untouched: TrainData is still {TrainData.shape}, using {selected_fps}.")

# ---------------------------------------------------------------------------
# TO ACTUALLY USE A FUSED SET for the rest of the notebook, replace BOTH matrices
# here and re-run everything from Section 4 onwards:
#
#     TrainData = TrainData_fused3
#     TestData = TestData_fused3
#
# Replace both, or neither. Training on fused features and predicting on plain
# ECFP4 (or the reverse) silently feeds the model the wrong columns.
#
# WARNING: expect this to be slow. Section 5's cross-validation and Section 7's
# ensemble both refit models repeatedly, and at 4,263 features instead of 2,048
# they take roughly twice as long.
#
# Note that Extra step 1 builds its own matrices per fingerprint via process_data(),
# so it is unaffected by this - it is already a form of using several fingerprints.
# ---------------------------------------------------------------------------

## Optional · Changing how much negative data you train on

This sample is balanced 50/50, which is convenient but not what a real DEL screen looks
like — there, inactives outnumber actives by orders of magnitude. When you point this
notebook at the full dataset you will have far more negatives than positives, and how many
of them you keep becomes a real decision.

The amount is set as a **ratio: negatives per positive.** `1.0` is balanced, `5.0` keeps
five inactives for every active, `0.5` keeps half as many inactives as actives. Every
positive is always kept, because actives are the scarce and expensive part of the data.

> **If you ask for more negatives than exist, you get all of them.** On this balanced
> sample there are only 2,000 inactives, so any ratio above 1.0 simply returns the full
> training data, and the cell says so. On the real dataset those higher ratios will do
> something.

> **Optional, and it works on a copy.** `resample_negatives` returns a new DataFrame;
> `df_train`, `TrainData` and `TrainLabel` are never modified. The cell prints both at the
> end so you can confirm it.

> **A trap in the results below.** Average precision *rises* as you remove negatives, which
> looks like an improvement and is not one. Average precision depends on how common the
> positive class is, so it cannot be compared across rows with different balances. AUC-ROC,
> which does not depend on the balance, drifts down slightly — the real signal that you
> have less data to learn from.

In [ ]:
# OPTIONAL - works on copies; df_train, TrainData and TrainLabel are untouched.
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold

from src.features import resample_negatives, balance_summary, stack_fingerprint

# Negatives per positive. 1.0 is balanced; above 1.0 needs more inactives than this
# balanced sample has, so those rows come back capped - on the real data they will not.
ratios = [0.25, 0.5, 1.0, 3.0]

print("What each requested ratio does to the training set:")
print(balance_summary(df_train, ratios).to_string(index=False))

print()
print("And what it does to 3-fold cross-validated performance:")
rows = []
for ratio in ratios:
    subset = resample_negatives(df_train, ratio=ratio)
    scores = cross_validate(
        LGBMClassifier(random_state=42, verbose=-1),
        stack_fingerprint(subset, selected_fps), subset["LABEL"],
        cv=StratifiedKFold(3, shuffle=True, random_state=42),
        scoring=["roc_auc", "average_precision"])
    rows.append({"ratio": ratio,
                 "rows used": len(subset),
                 "AUC-ROC": round(float(scores["test_roc_auc"].mean()), 3),
                 "avg precision": round(float(scores["test_average_precision"].mean()), 3)})

print(pd.DataFrame(rows).to_string(index=False))

print()
print(f"df_train unchanged : {len(df_train)} rows, {int(df_train['LABEL'].sum())} actives")
print(f"TrainData unchanged: {TrainData.shape}")

# ---------------------------------------------------------------------------
# TO ACTUALLY TRAIN ON A REBALANCED SET, do it explicitly and re-run from Section 4:
#
#     subset = resample_negatives(df_train, ratio=5.0)
#     TrainData = stack_fingerprint(subset, selected_fps)
#     TrainLabel = subset["LABEL"]
#
# Only the TRAINING data is ever rebalanced. Never resample the test set - its class
# balance is the thing you are trying to measure performance against.
# ---------------------------------------------------------------------------

## 💬 Discussion · Dimensionality reduction

> **The idea.** 2,048 bits per molecule, only about 3% of them non-zero. Compressing that
> into a few hundred dense components — PCA, or `TruncatedSVD`, which handles sparse data
> without centring it — is an obvious thing to reach for.
>
> **Why you might**
>
> - Fewer, denser features train faster and need less memory.
> - It can suppress noise from bits that fire more or less at random.
>
> **Why it often disappoints here**
>
> - Gradient boosting already ignores useless features: it simply never splits on them. Width
>   hurts distance-based methods far more than it hurts trees.
> - You lose the chemistry. A fingerprint bit means "this substructure is present"; component
>   17 means nothing you can show a chemist or turn into a design decision.
> - `UMAP` and `t-SNE` are for **looking** at data, not for feeding models. They distort
>   global distances, and t-SNE cannot even transform new molecules.
>
> **Whatever you use, fit it on the training folds only** and apply it to validation and test.
> Fitting on everything first leaks information and inflates your score — the same trap as
> scaling before the split.

## 💬 Discussion · Feature selection

> **The idea.** Rather than compressing the bits, throw some away. Drop the ones that are
> constant or nearly so, or keep the top few hundred by model importance or univariate score.
>
> **Why you might**
>
> - Faster training, a smaller model, and a shortlist of substructures you can actually look
>   at and reason about chemically.
> - Bits that never fire anywhere in your library carry no information at all.
>
> **Why it is riskier than it looks**
>
> - Selection is part of training. Choose your features using all the data — validation folds
>   included — and the score you get afterwards is fiction. It has to happen **inside** each fold.
> - A rare bit is not a useless bit. The substructure defining a hit series might appear in
>   nine molecules out of five thousand, and a frequency cut-off will delete it.
> - With only a handful of actives, importance rankings are unstable. Re-run with a different
>   seed and check the same bits survive before you trust them.
>
> Both of these are worth an experiment, but neither is free — and on tabular fingerprint data
> a well-tuned model on the raw bits is a genuinely hard baseline to beat.

---

# ⚙️ Section 4 · Define the ML Model

## First: are we predicting a number or a category?

- **Regression** predicts a *number*. How enriched is this compound? What sequencing count
  would we expect?
- **Classification** predicts a *category*, normally with a probability attached. Is this
  compound active, yes or no?

Our training file carries both. `RawCount` is the raw sequencing readout, and `LABEL` is
`1` exactly when that count is above zero — actives here run from 6 counts up to 247.

**We classify**, for two reasons. The label is what a chemist acts on: you either put a
compound on the plate or you do not. And counts are noisy — the gap between 6 and 9 reads
says much more about sequencing depth than about binding.

> **When regression would be the better choice:** if you trust your enrichment values, a
> regression model ranks the strong binders above the marginal ones instead of lumping them
> into one "active" bucket. Note that only the training file has `RawCount` here, so we
> could not score such a model on the test set anyway.
>
> Almost everything else in this notebook — the splitting, the cross-validation, the
> ranking, the screen — would stay exactly the same. Only the model class and the metrics
> change.

## Which kind of classifier?

Every classifier draws a boundary between actives and inactives. They differ in the *shape*
of boundary they are able to draw, and in what that costs.

| Family | Example | The idea in one line | Watch out for |
|---|---|---|---|
| **Linear** | `LogisticRegression` | weigh each feature, add them up, squash the total into a probability | can only draw a straight boundary; wants scaled inputs |
| **Margin-based** | `SVC` | find the boundary with the widest possible gap between the classes | scales badly — slow well before 4,000 × 2,048 |
| **Bagged trees** | `RandomForestClassifier` | grow hundreds of trees on random slices and average them | large models, and rarely the strongest on this kind of data |
| **Neural network** | `MLPClassifier` | stacked weighted sums with non-linearities between them | hungry for data and tuning; seldom beats boosting on tabular data |

The fifth family is **boosted trees** — trees grown one at a time, each correcting the
errors of those before it. That is LightGBM, and it is what we will use.

The next cell defines all four — one line each, showing the few settings you would actually
reach for. Every value shown is the library default, so they behave exactly as the bare
constructor would. The end of the cell says where to plug one in.

In [ ]:
# Four alternatives to LightGBM, one line each. The values shown are the library
# defaults, so these behave exactly as `LogisticRegression()` would - they are spelled
# out only so you can see which knobs exist. Nothing is fitted in this cell.
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

logistic_model = LogisticRegression(C=1.0, penalty="l2", max_iter=1000, random_state=42)
svm_model = SVC(C=1.0, kernel="rbf", gamma="scale", probability=True, random_state=42)
forest_model = RandomForestClassifier(n_estimators=100, max_depth=None, n_jobs=-1, random_state=42)
mlp_model = MLPClassifier(hidden_layer_sizes=(100,), alpha=1e-4, max_iter=200, random_state=42)

alternative_models = {
    "Logistic regression": logistic_model,    # C: smaller means a simpler model
    "Support vector machine": svm_model,      # C and kernel; "linear" is far faster
    "Random forest": forest_model,            # n_estimators, max_depth
    "Neural network (MLP)": mlp_model,        # hidden_layer_sizes, e.g. (256, 64)
}

for name, estimator in alternative_models.items():
    print(f"{name:<24} {type(estimator).__name__}")

print()
print("All four answer to .fit(X, y) and .predict_proba(X), which is what makes")
print("swapping one in a small change rather than a rewrite.")

# To use one instead of LightGBM, replace the constructor in the two places a model is
# built - train_and_validate() in Section 5, train_final_model() in Section 7 - then
# re-run from there. Change both, so what you validate is what you screen with.
#
# On this data (4,000 x 2,048): RandomForest runs fine and is a fair baseline for
# LightGBM. LogisticRegression works but wants scaled features. SVC and MLP are slow
# at this width and also want scaling.

## LightGBM

Light Gradient Boosting Machine is a fast, scalable gradient boosting framework. It builds
decision trees iteratively, each one correcting the errors of the ones before it. Unlike
older gradient boosting implementations it uses a histogram-based approach, which speeds up
training considerably on large datasets, and it handles categorical features directly.

> **Why it suits this problem.** Fingerprints are wide, sparse, tabular count vectors, and
> gradient boosting is consistently the strongest family on tabular data. This is the same
> reasoning laid out in Section 2 of the ML introduction notebook.

In [ ]:
from lightgbm import LGBMClassifier

# Initialize model with detailed hyperparameters using default values
model = LGBMClassifier(
    n_estimators=100,  # Number of boosting iterations (trees)
    n_jobs=1,  # Number of parallel jobs (1 for no parallelism)
    learning_rate=0.1,  # Learning rate
    max_depth=-1,  # No limit on maximum depth of trees
    min_child_samples=20,  # Minimum samples at leaf node
    reg_lambda=0.0,  # L2 regularization (no regularization)
    reg_alpha=0.0,  # L1 regularization (no regularization)
    num_leaves=31,  # Number of leaves in each tree
    max_bin=255,  # Maximum number of bins
    subsample=1.0,  # Subsample ratio for training data (use all data)
    colsample_bytree=1.0,  # Subsample ratio for features (use all features)
    random_state=42,  # Random seed for reproducibility
    boosting_type='gbdt',  # Boosting type (Gradient Boosting Decision Tree)
    min_split_gain=0.0,  # Minimum loss reduction required to make a further partition
    verbose=-1,  # Quieten LightGBM's per-tree logging
)

# Model is now initialized with default hyperparameters

---

# 📏 Metrics · What Are We Actually Measuring?

Before we train anything, we should agree on how we will judge it. Pick the wrong metric and
you will confidently choose the wrong model.

Every classification metric is built from four counts. For one compound the model either
flags it or it does not, and it either is active or it is not:

|  | Predicted inactive | Predicted active |
|---|---|---|
| **Actually inactive** | true negative | **false positive** — an assay run for nothing |
| **Actually active** | **false negative** — a hit we never tested | true positive |

The two errors do not cost the same. A false positive wastes one well on a plate. A false
negative may be the compound the whole campaign was looking for.

## Metrics that need a yes/no decision

These take the model's hard prediction, which means they depend on where you put the
threshold. All of them appear in the cross-validation output in the next section.

| Metric | What it asks | Blind spot |
|---|---|---|
| **Accuracy** | what fraction did I get right? | useless when one class is rare — see below |
| **Precision** | of the ones I flagged, how many were real? | says nothing about the hits you missed |
| **Recall** | of the real actives, how many did I find? | flagging everything gives perfect recall |
| **F1** | the balance between precision and recall | one number hides which of the two is failing |
| **MCC** | agreement across all four counts at once | harder to explain to a non-specialist |
| **Cohen's kappa** | how much better than guessing at the same rate? | same idea as MCC, slightly different scale |

MCC and Cohen's kappa are the trustworthy single numbers on imbalanced data: both stay near zero
for a model that is really just exploiting the class balance.

## Metrics that use the ranking

These ignore the threshold and look at the ordering, which is closer to what a screen does.

| Metric | What it asks |
|---|---|
| **ROC-AUC** | pick one active and one inactive at random — how often is the active scored higher? |
| **Average precision (PR-AUC)** | precision averaged across every recall level |
| **hit@K** | of the top K compounds, how many are actually active? |
| **Enrichment** | how many times better than picking K at random? |

## Which ones matter for this workshop

Our test set has **9 actives in 5,000 compounds**, and we will only ever assay the top of the
list. That combination decides everything:

- **Ignore accuracy.** A model that calls everything inactive scores 99.8%.
- **ROC-AUC flatters.** With 4,991 inactives dominating it, it stays high even when the top
  of the list is poor.
- **Trust average precision, hit@K and enrichment.** They ask about the rare positives and
  about the part of the list you can afford to test — which is the actual decision.

The next cell shows all of this on made-up numbers, before any real model exists.

In [ ]:
# A demonstration on INVENTED data - no model is trained here. We fake three
# rankings over 5,000 compounds with 9 actives, matching our real test set, and
# see how each metric reacts.
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             matthews_corrcoef, cohen_kappa_score,
                             roc_auc_score, average_precision_score)

rng = np.random.default_rng(7)
n_compounds, n_actives = 5000, 9

# Scatter the actives at random positions. If they sat at the top of the array,
# a model that scores everything the same would appear to "find" them.
y_mock = np.zeros(n_compounds, dtype=int)
y_mock[rng.choice(n_compounds, n_actives, replace=False)] = 1

# three different "models", none of them real
scores = {
    "Calls everything inactive": np.full(n_compounds, 0.01),
    "Random ranking": rng.random(n_compounds),
    "A decent model": np.where(y_mock == 1,
                               rng.normal(0.70, 0.18, n_compounds),
                               rng.normal(0.20, 0.12, n_compounds)).clip(0, 1),
}

print(f"{n_actives} actives in {n_compounds} compounds "
      f"({n_actives / n_compounds:.2%}) - the same balance as our test set\n")

header = f"{'model':<26}{'accuracy':>10}{'precision':>11}{'recall':>8}{'F1':>7}{'MCC':>7}{'ROC-AUC':>9}{'PR-AUC':>8}{'hit@20':>8}"
print(header)
print("-" * len(header))

for name, s in scores.items():
    pred = (s >= 0.5).astype(int)                 # the yes/no decision
    top20 = np.argsort(-s, kind="stable")[:20]    # the ranking
    print(f"{name:<26}"
          f"{accuracy_score(y_mock, pred):>10.4f}"
          f"{precision_score(y_mock, pred, zero_division=0):>11.3f}"
          f"{recall_score(y_mock, pred, zero_division=0):>8.3f}"
          f"{f1_score(y_mock, pred, zero_division=0):>7.3f}"
          f"{matthews_corrcoef(y_mock, pred):>7.3f}"
          f"{roc_auc_score(y_mock, s):>9.3f}"
          f"{average_precision_score(y_mock, s):>8.3f}"
          f"{int(y_mock[top20].sum()):>8d}")

print()
print("Look down the accuracy column. Calling everything inactive scores 0.9982,")
print("which is HIGHER than the decent model - the only one of the three worth")
print("using. Every other column disagrees, and those are the ones telling the truth.")
print()
print("Compare ROC-AUC with PR-AUC as well. The random ranking sits near 0.5 on")
print("ROC, which sounds merely unimpressive, and near 0.00 on PR-AUC, which")
print("sounds like the disaster it is. Both numbers are correct.")

### The same point, drawn

Two curves for the invented "decent model". They describe the identical predictions.

The **ROC** curve hugs the top-left and reports a flattering number, because with 4,991
inactives the false-positive rate barely moves however many mistakes you make. The
**precision-recall** curve tells the harder truth: to find all 9 actives you must accept a
lot of dead weight. The dashed line on the PR plot is what random picking achieves — note how
far below the axis it sits, which is exactly why a rare positive class makes PR the useful
view.

The **enrichment curve** answers the question a screener actually asks: if I can afford to
test the top few percent, what share of the hits do I get?

In [ ]:
# Still invented data - same helpers we will use on the real model later.
plot_pr_and_roc(y_mock, scores["A decent model"])
plt.show()

plot_enrichment_curve(y_mock, scores["A decent model"], zoom_frac=0.1)
plt.show()

print("Keep the shape of these in mind. The same two plots appear in Section 7,")
print("drawn from a model that was actually trained.")

---

# 🔁 Section 5 · Training the Model and Evaluating on Cross-Validation

## Cross-validation

We need to know how good the model is **before** we are allowed to open the test set. So we
work entirely inside the training data: cut it into five equal parts, called *folds*, and go
round five times.

```
round 1   [validate]   train      train      train      train
round 2    train      [validate]   train      train      train
round 3    train       train      [validate]   train      train
round 4    train       train       train      [validate]   train
round 5    train       train       train       train      [validate]
```

Each round fits a **fresh** model on four folds and scores it on the fifth. Every compound in
the training data is used for fitting four times, and for validation exactly once — so we get
five independent estimates instead of one lucky or unlucky split.

Note the vocabulary carefully: the held-out fold is a **validation fold**. It is not a test
set. The test set is a different file, and we have not opened it yet.

> **These numbers will look better than screening reality, and that is expected.** The
> training data is balanced 50/50, so a validation fold is too. Half of everything is active,
> which makes almost any prediction look good — watch `Hits@20` come back as 20 out of 20
> below. Section 7 is where the real numbers arrive.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, cohen_kappa_score
)


# Fit on the training folds, score on the held-out VALIDATION fold.
# Nothing in this cell touches the test data.
def train_and_validate(X_fold_train, X_fold_val, y_fold_train, y_fold_val):
    """Train a LightGBM model on the training folds and score it on the validation fold."""
    model = LGBMClassifier(random_state=42, verbose=-1)
    model.fit(X_fold_train, y_fold_train)

    y_pred = model.predict(X_fold_val)
    y_scores = model.predict_proba(X_fold_val)[:, 1]  # Probability for positive class

    metrics = {
        "Accuracy": accuracy_score(y_fold_val, y_pred),
        "Precision": precision_score(y_fold_val, y_pred, zero_division=0),
        "Recall": recall_score(y_fold_val, y_pred),
        "F1-Score": f1_score(y_fold_val, y_pred),
        "AUC-ROC": roc_auc_score(y_fold_val, y_scores) if len(set(y_fold_val)) > 1 else None,
        "MCC": matthews_corrcoef(y_fold_val, y_pred),
        "Cohen's Kappa": cohen_kappa_score(y_fold_val, y_pred),
    }

    # Hit@K: of the K highest-scoring compounds in this validation fold, how many are
    # active? Ks larger than the fold are skipped automatically.
    for k, hits in hits_at_k(y_fold_val, y_scores, ks=KS).items():
        metrics[f"Hits@{k}"] = hits

    return model, metrics


# Five-fold cross-validation, entirely within the training data
Nfold = 5
skf = StratifiedKFold(n_splits=Nfold, shuffle=True, random_state=42)
fold_metrics = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(TrainData, TrainLabel)):
    # Four folds train the model, the fifth validates it
    X_fold_train, X_fold_val = TrainData[train_idx], TrainData[val_idx]
    y_fold_train = TrainLabel.iloc[train_idx]
    y_fold_val = TrainLabel.iloc[val_idx]

    _, metrics = train_and_validate(X_fold_train, X_fold_val, y_fold_train, y_fold_val)
    fold_metrics.append(metrics)

    print(f"Fold {fold_idx + 1} - validation metrics "
          f"({len(y_fold_train)} training rows, {len(y_fold_val)} validation rows):")
    for metric, value in metrics.items():
        # hit counts are integers; everything else is a rate
        print(f"{metric}: {value:d}" if metric.startswith("Hits@")
              else f"{metric}: {value:.4f}")
    print("-" * 100)

In [ ]:
# Average the validation metrics across the five folds
avg_metrics = {metric: np.mean([fold[metric] for fold in fold_metrics])
               for metric in fold_metrics[0]}

print("\nAverage validation metrics across all folds:")
for metric, value in avg_metrics.items():
    print(f"{metric}: {value:.1f}" if metric.startswith("Hits@")
          else f"{metric}: {value:.4f}")

# The validation folds are balanced 50/50, so hit@K comes out close to K - almost any
# compound you pick is active. That is exactly why these numbers say nothing about
# screening performance. The test set in Section 7 is where reality arrives.
plot_cv_metrics(fold_metrics)
plt.show()

## 💬 Discussion · Is this cross-validation telling us the truth?

> **The scores above look good. Be suspicious of them.**
>
> `StratifiedKFold(shuffle=True)` scatters molecules across the folds at random. That is the
> right default when rows are independent of each other. DEL rows are not.
>
> A DNA-encoded library is **combinatorial**. Every compound is built by joining a handful of
> building blocks, one per synthesis cycle, and the `DEL_ID` column records exactly which
> ones: `L30-419-410-879` is library 30 carrying blocks 419, 410 and 879.
>
> Count them in our training data and the problem is hard to miss. Those 4,000 molecules use
> only **645 distinct cycle-1 blocks**, and **3,790 of the 4,000 share a cycle-1 block with at
> least one other molecule**. A single block turns up in 287 of them.
>
> So a random split all but guarantees that a molecule sitting in a validation fold has close
> relatives in the training folds — same core, same substituent, different in one cycle. The
> model never has to generalise. It can recognise a building block it was already paid to
> memorise, and collect the score.
>
> **What the number actually means.** Our validation score measures performance on *new
> combinations of familiar chemistry*. That is a real question and sometimes the one you want.
> It is not the question "will this work on a library we have never seen before?" — and that
> second question is usually the one deciding whether the model is worth anything.
>
> ---
>
> **Worth thinking about — we are deliberately not answering these:**
>
> - What would a split have to guarantee before its score could mean "works on unfamiliar chemistry"?
> - How would you decide that two molecules are *too* similar to sit on opposite sides of a split? A shared building block? A shared scaffold? A Tanimoto similarity above some cut-off — and above which cut-off?
> - `DEL_ID` is sitting right there in the training data, already parsed into cycles. What could you group on, and at what level?
> - Extra step 3 clusters molecules by structural similarity to pick a *diverse shortlist*. Could that same clustering do a different job much earlier in this notebook?
> - `scikit-learn` has `GroupKFold`, which keeps every member of a group on one side of the split. What would you hand it as `groups`?
> - What does a stricter split cost? Fewer usable folds, a smaller effective training set, noisier estimates — and a lower number to report. If your score falls when you split this way, which number goes in the slide deck?
>
> None of these has one correct answer, and the trade-offs are argued about in the literature
> to this day. Pick an approach, run it against the random split above, and find out how much
> of your performance was real.

---

# 🎛️ Section 6 · Hyperparameter Tuning

Some numbers a model *learns* from data. Others you have to *choose* before training starts —
how many trees, how fast each one corrects the last, how deep they grow. Those are
**hyperparameters**, and LightGBM's defaults are reasonable rather than right for your data.

## How people search

| Method | How it works | When to reach for it |
|---|---|---|
| **Grid search** | try every combination in a grid you define | few parameters, and you want exhaustive coverage |
| **Random search** | sample combinations at random from ranges you give | many parameters — usually finds something good faster than a grid, because most parameters do not matter much |
| **Bayesian optimisation** | build a model of which settings did well, then try where it predicts improvement | each fit is expensive and you can only afford a few dozen |

`scikit-learn` ships `GridSearchCV` and `RandomizedSearchCV`, both of which slot straight
around any estimator. Bayesian search needs a separate library — **Optuna** is the usual
choice, with `scikit-optimize` and `hyperopt` as alternatives.

## What we do here

A small hand-picked grid of **10 configurations**, each scored by 5-fold cross-validation on
the training data. That is 50 model fits and takes about a minute.

Everything stays inside the training data. Tuning is a decision, and decisions are exactly
what the test set must not see — otherwise the estimate we get in Section 7 is no longer
trustworthy.

**We select on hit@200.** Not accuracy, not AUC. The whole point of this model is to put
actives near the top of a ranked list, so the configuration we want is the one whose ranking
packs the most actives into its first 200 compounds. Pick the metric that matches the
decision you will actually make.

> **Two caveats worth stating.**
>
> The validation folds are balanced 50/50, so hit@200 crowds up near its ceiling of 200 and
> the configurations end up separated by only a few compounds. On genuinely imbalanced data
> it discriminates far more sharply. AUC and average precision are printed alongside so you
> can see whether they agree with the ranking.
>
> Ten hand-written configurations is a demonstration, not a real search. For real work, hand
> the same grid to `RandomizedSearchCV` with a few hundred samples, or to Optuna, and let it
> explore properly.

In [ ]:
# Ten configurations, scored by 5-fold cross-validation on the TRAINING data only.
# About a minute: 10 configs x 5 folds = 50 model fits.
from sklearn.metrics import roc_auc_score, average_precision_score

SELECT_K = 200  # we keep the configuration that puts the most actives in its top 200

param_grid = [
    {"n_estimators": 100, "learning_rate": 0.10, "num_leaves": 31},   # library defaults
    {"n_estimators": 300, "learning_rate": 0.10, "num_leaves": 31},   # more trees
    {"n_estimators": 300, "learning_rate": 0.05, "num_leaves": 31},   # slower learning
    {"n_estimators": 600, "learning_rate": 0.05, "num_leaves": 31},   # slower, more trees
    {"n_estimators": 300, "learning_rate": 0.10, "num_leaves": 63},   # bushier trees
    {"n_estimators": 300, "learning_rate": 0.10, "num_leaves": 15},   # simpler trees
    {"n_estimators": 300, "learning_rate": 0.10, "num_leaves": 31, "min_child_samples": 5},
    {"n_estimators": 300, "learning_rate": 0.10, "num_leaves": 31, "colsample_bytree": 0.5},
    {"n_estimators": 300, "learning_rate": 0.10, "num_leaves": 31, "reg_lambda": 10.0},
    {"n_estimators": 600, "learning_rate": 0.05, "num_leaves": 63, "colsample_bytree": 0.5},
]

tuning_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
for config_id, params in enumerate(param_grid, start=1):
    hits, aucs, aps = [], [], []

    for train_idx, val_idx in tuning_cv.split(TrainData, TrainLabel):
        candidate = LGBMClassifier(random_state=42, verbose=-1, n_jobs=-1, **params)
        candidate.fit(TrainData[train_idx], TrainLabel.iloc[train_idx])

        y_val = TrainLabel.iloc[val_idx]
        scores = candidate.predict_proba(TrainData[val_idx])[:, 1]

        hits.append(hits_at_k(y_val, scores, ks=[SELECT_K])[SELECT_K])
        aucs.append(roc_auc_score(y_val, scores))
        aps.append(average_precision_score(y_val, scores))

    results.append({"config": config_id, **params,
                    f"Hits@{SELECT_K}": round(float(np.mean(hits)), 1),
                    "AUC-ROC": round(float(np.mean(aucs)), 4),
                    "avg precision": round(float(np.mean(aps)), 4)})
    print(f"  config {config_id:>2} of {len(param_grid)} done")

results_df = (pd.DataFrame(results)
              .fillna("-")
              .sort_values(f"Hits@{SELECT_K}", ascending=False))

print()
print(f"Ranked by mean Hits@{SELECT_K} across the five validation folds:")
print(results_df.to_string(index=False))

# The winner, and the settings we will carry into Section 7
best_config = int(results_df.iloc[0]["config"])
best_params = param_grid[best_config - 1]
best_score = results_df.iloc[0][f"Hits@{SELECT_K}"]
default_score = results_df.loc[results_df["config"] == 1, f"Hits@{SELECT_K}"].iloc[0]

print()
print(f"Best configuration : #{best_config}   Hits@{SELECT_K} = {best_score}")
print(f"  {best_params}")
print(f"Library defaults   : #1   Hits@{SELECT_K} = {default_score}")
print()
print("Section 7 trains the final model with these settings.")

---

# 📊 Section 7 · Train the Final Model and Test It

Cross-validation showed the approach works; Section 6 chose the settings. Every decision is
now made, so we do two things.

First, fit **one** model on all 4,000 training molecules — nothing held back, because there
are no more choices to inform. Second, score it on the test set.

This is the first time `df_test` is touched. Because nothing about it influenced the model,
the number it gives us is unbiased.

In [ ]:
import joblib
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score


# Fit one model on ALL the training data, using the settings Section 6 chose.
def train_final_model(X, y, params):
    final_model = LGBMClassifier(random_state=42, verbose=-1, n_jobs=-1, **params)
    final_model.fit(X, y)
    model_filename = RESULTS_DIR / "final_model.pkl"
    joblib.dump(final_model, model_filename)
    print(f"Final model saved as {model_filename}.")
    return final_model


print(f"Training the final model with the tuned settings: {best_params}")
final_model = train_final_model(TrainData, TrainLabel, best_params)


def report(name, X, y):
    """Print the classifier view, then the ranking view that actually matters."""
    pred = final_model.predict(X)
    scores = final_model.predict_proba(X)[:, 1]

    print(f"\n{name}  ({int(y.sum())} actives out of {len(y)})")
    print(f"  Accuracy:          {accuracy_score(y, pred):.4f}")
    print(f"  AUC-ROC:           {roc_auc_score(y, scores):.4f}")
    print(f"  Average precision: {average_precision_score(y, scores):.4f}")
    print(f"  Predicting 'inactive' for everything would score "
          f"{accuracy_score(y, np.zeros_like(y)):.4f} accuracy")

    # You never assay a whole library - you assay the top of the ranking. So the
    # question is: if we could afford K assays, how many real hits would we get?
    print(f"\n  If we could only afford to test the top K compounds:")
    print("  " + screening_table(y, scores, ks=KS).replace("\n", "\n  "))
    return scores


# The clean estimate: 9 actives among 5,000 compounds, and every modelling choice
# was already made using the training data and its validation folds.
scores_test = report("Test set (sample-test)", TestData, TestLabel)

## What the model actually learned

Three views of the same test-set predictions. Together they say more than any single number.

- **Score distribution** — the actives should sit to the right of the inactive bulk. The
  count axis is logarithmic, because 9 actives against 4,991 inactives would otherwise be
  invisible.
- **Precision-recall and ROC** — with 0.18% positives, trust the precision-recall curve.
  ROC looks flattering here because the huge pool of true negatives dominates it.
- **Enrichment curve** — the practical one. It answers "if I screen the top 2% of the
  library, what fraction of the hits do I get?" The dashed diagonal is what random picking
  would give you.

In [ ]:
# Where do the actives sit in the score distribution?
plot_score_distribution(TestLabel, scores_test,
                        title="Test-set predicted score by true class")
plt.show()

# Precision-recall is the informative curve when positives are rare; ROC is alongside.
plot_pr_and_roc(TestLabel, scores_test)
plt.show()

# How many actives do we recover as we screen deeper into the ranked test library?
plot_enrichment_curve(TestLabel, scores_test, zoom_frac=0.1)
plt.show()

---

# 🔬 Virtual Screening

Sections 1 to 7 built a model, tuned it and tested it. From here we put it to use: score
a library of compounds, combine several fingerprints for robustness, then filter and cluster
what comes out into a shortlist.

---

# 🎯 Section 8 · Screen the Test Compounds

This is the first time we touch `df_test`. Every decision — fingerprint, model,
hyperparameters — has already been made using the training data and its validation folds, so
whatever we get here is a fair estimate rather than a number we tuned towards.

Scoring these compounds *is* the virtual screen: we rank them by predicted activity and
nominate the top of the list.

In [ ]:
def evaluate_model(model, X):
    """Score a set of compounds and return the probability of being active."""
    y_scores = model.predict_proba(X)[:, 1]  # Probability for positive class
    return np.round(y_scores, 3)


predictions = evaluate_model(final_model, TestData)

# Create a DataFrame with SMILES and prediction scores
prediction_df = pd.DataFrame({
    'SMILES': df_test["SMILES"],
    'Prediction_Score': predictions
})

# Sort the DataFrame by prediction score in descending order
prediction_df_sorted = prediction_df.sort_values(by='Prediction_Score', ascending=False)

# Keep only those with score > 0.5 as Possible Nominees
nominees = prediction_df_sorted[prediction_df_sorted['Prediction_Score'] > 0.5]

# Get the number of nominees
num_nominees = nominees.shape[0]
print(f"\nNumber of Possible Nominees: {num_nominees}")

# Because this test set is labelled, we can check the screen against the truth.
# A genuine screening library cannot be scored this way - that is what makes it genuine.
print(f"\nSingle model ({selected_fps}) - hits among the top K:")
print(screening_table(TestLabel, predictions, ks=KS))

# Print the top 10 highest-ranked predictions
print("\nTop 10 Predictions:")
print(prediction_df_sorted.head(10))

---

# 🧭 Extra Steps · Optional

Sections 1 to 8 did the work. What follows does not change your submission — **Section 9, at
the end of this notebook, is still your deliverable.**

These steps show what a real campaign does with a ranked list before anyone touches a plate:
combine several models so no single fingerprint decides everything, filter out compounds
unlikely to behave as drugs, and pick a structurally diverse shortlist rather than twenty
variations on one scaffold.

Run them now if you have time, skip straight to Section 9 if not.

---

# 🧩 Extra step 1 · Using an Ensemble of Models

This improves reliability by training one model per fingerprint (ECFP4, MACCS, RDK) instead
of relying on a single representation. For each compound we take the mean prediction across
models, their standard deviation, and a confidence score.

The final score is **mean minus standard deviation**, so compounds the models disagree about
are pushed down the list. That is deliberately conservative — see the threshold comparison
below for what it costs you.

In [ ]:
import numpy as np
import pandas as pd
import joblib
from lightgbm import LGBMClassifier

# List of fingerprint columns to train on
fingerprint_columns = ['ECFP4', 'ECFP6', 'FCFP4', 'FCFP6', 'MACCS', 'RDK', 'AVALON', 'ATOMPAIR', 'TOPTOR']
fingerprint_columns = ['ECFP4', 'MACCS', 'RDK']

# Dictionary to store trained models
trained_models = {}

# Train models for each fingerprint column
for fp in fingerprint_columns:
    print(f"Training model for {fp}...")
    TrainData_fp = process_data(df_train, fp)

    model = LGBMClassifier(random_state=42, verbose=-1)
    model.fit(TrainData_fp, TrainLabel)

    model_filename = RESULTS_DIR / f"final_model_{fp}.pkl"
    joblib.dump(model, model_filename)
    print(f"Model for {fp} saved as {model_filename}.")

    trained_models[fp] = model


# Function to evaluate models
def evaluate_model(model, X):
    return model.predict_proba(X)[:, 1]  # Probability for positive class


# Store predictions for each model
all_predictions = {}

for fp in fingerprint_columns:
    print(f"Evaluating model for {fp}...")
    TestData_fp = process_data(df_test, fp)
    all_predictions[fp] = evaluate_model(trained_models[fp], TestData_fp)

# Convert predictions to DataFrame
predictions_df = pd.DataFrame(all_predictions, index=df_test.index)
predictions_df['Mean_Prediction'] = predictions_df[fingerprint_columns].mean(axis=1)
predictions_df['Std_Dev'] = predictions_df[fingerprint_columns].std(axis=1)
predictions_df['Confidence_Score'] = 1 - predictions_df['Std_Dev']  # Higher means more confident
predictions_df['Final_Score'] = predictions_df['Mean_Prediction'] - predictions_df['Std_Dev']

# Add SMILES, plus the true label so the shortlist can be checked at the end.
# LABEL is never used as a feature - it only lets us verify the screen afterwards,
# which is possible because this test set happens to be labelled.
predictions_df['SMILES'] = df_test['SMILES']
predictions_df['LABEL'] = df_test['LABEL']

# Sort by Final Score
predictions_df_sorted = predictions_df.sort_values(by='Final_Score', ascending=False)

# Select nominees with Final Score > 0.5
nominees = predictions_df_sorted[predictions_df_sorted['Final_Score'] > 0.5]
num_nominees = nominees.shape[0]

print(f"\nNumber of Possible Nominees: {num_nominees}")
print("Top 10 Predictions:")
print(predictions_df_sorted.head(10))

In [ ]:
# The ensemble is a different ranking, so score it the same way as the single model.
print(f"Ensemble of {fingerprint_columns} - hits among the top K:")
print(screening_table(TestLabel, predictions_df["Final_Score"], ks=KS))

single = hits_at_k(TestLabel, predictions, ks=KS)
ens = hits_at_k(TestLabel, predictions_df["Final_Score"], ks=KS)
print("\nSingle model vs ensemble, hits at each K:")
print(f"  {'K':>6} {'single':>8} {'ensemble':>9}")
for k in single:
    print(f"  {k:>6} {single[k]:>8} {ens[k]:>9}")

# ---------------------------------------------------------------------------
# How many nominees does each threshold give, and how many real actives survive?
# ---------------------------------------------------------------------------
known_actives = int(TestLabel.sum())
thresholds = [0.5, 0.4, 0.3, 0.2]
n_selected, actives_kept = [], []

print(f"\n{'threshold':>10}  {'nominees':>9}  actives kept (of {known_actives})")
for thr in thresholds:
    selected = predictions_df_sorted["Final_Score"] > thr
    kept = int(df_test.loc[predictions_df_sorted.index[selected], "LABEL"].sum())
    n_selected.append(int(selected.sum()))
    actives_kept.append(kept)
    print(f"{thr:>10.1f}  {int(selected.sum()):>9d}  {kept}")

plot_threshold_tradeoff(thresholds, n_selected, actives_kept, known_actives)
plt.show()

# Final_Score is mean-minus-std, so it punishes disagreement between fingerprints.
# A strict cut-off is not free: it discards real hits the models argued about.
THRESHOLD = 0.3
nominees = predictions_df_sorted[predictions_df_sorted["Final_Score"] > THRESHOLD]

print(f"\nCarrying {nominees.shape[0]} nominees forward at Final_Score > {THRESHOLD}.")
print("Top 10:")
print(predictions_df_sorted.head(10))

---

# 🧪 Extra step 2 · Apply Medicinal Chemistry Filters

This part applies drug-likeness filters to the nominees, using three standard rules. Molecules
passing all three are kept as final candidates.

- **Lipinski’s Rule of 5** — molecular weight, lipophilicity (logP), hydrogen bond
  donors and acceptors, and rotatable bonds.
- **Ghose filter** — molecular weight, logP, atom count and molar refractivity, for
  favourable pharmacokinetics.
- **Veber rule** — limited rotatable bonds and acceptable topological polar surface area,
  for oral bioavailability.

> **Note:** this is a simplified version of a much larger family of drug-likeness filters.

In [ ]:
from rdkit import Chem
import rdkit.Chem.Descriptors as Descriptors

class SimplifiedDrugFilters:
    def __init__(self):
        pass

    @staticmethod
    def fetch_attributes(molecule):
        return {
            "molecular_weight": Descriptors.ExactMolWt(molecule),
            "logp": Descriptors.MolLogP(molecule),
            "h_bond_donor": Descriptors.NumHDonors(molecule),
            "h_bond_acceptors": Descriptors.NumHAcceptors(molecule),
            "rotatable_bonds": Descriptors.NumRotatableBonds(molecule),
            "num_atoms": Chem.rdchem.Mol.GetNumAtoms(molecule),
            "molar_refractivity": Chem.Crippen.MolMR(molecule),
            "topo_surface_area": Chem.QED.properties(molecule).PSA
        }

    def filter(self, smiles):
        results = {"lipinski": [], "ghose": [], "veber": [], "pass_all_filters": []}
        molecules = [Chem.MolFromSmiles(i) for i in smiles]

        for i, mol in enumerate(molecules):
            props = self.fetch_attributes(mol)

            # Lipinski Rule of 5
            lipinski = (props["molecular_weight"] <= 500 and props["logp"] <= 5 and
                        props["h_bond_donor"] <= 5 and props["h_bond_acceptors"] <= 10 and
                        props["rotatable_bonds"] <= 5)

            # Ghose Filter
            ghose = (160 <= props["molecular_weight"] <= 480 and -0.4 <= props["logp"] <= 5.6 and
                     20 <= props["num_atoms"] <= 70 and 40 <= props["molar_refractivity"] <= 130)

            # Veber Rule
            veber = (props["rotatable_bonds"] <= 10 and props["topo_surface_area"] <= 140)

            results["lipinski"].append(lipinski)
            results["ghose"].append(ghose)
            results["veber"].append(veber)
            results["pass_all_filters"].append(all([lipinski, ghose, veber]))

        return results

In [ ]:
# Apply drug design filters to the selected nominees
filter = SimplifiedDrugFilters()
filter_results = pd.DataFrame(filter.filter(nominees["SMILES"].tolist()), index=nominees.index)

# Merge the filter results with nominees
nominees_filtered = pd.merge(nominees, filter_results, left_index=True, right_index=True)
print("Top 10 Predictions:")
print(nominees_filtered.head(10))

nominees_filtered = nominees_filtered[nominees_filtered["pass_all_filters"] == True]
num_nominees_filtered = nominees_filtered.shape[0]
print(f"\nNumber of Filtered Nominees: {num_nominees_filtered}")

---

# 🗂️ Extra step 3 · Cluster and Select Compounds for Laboratory Testing

Finally, we use similarity-based clustering to pick a *diverse* shortlist from the top hits.
Nominating twenty near-identical molecules wastes twenty assays; nominating twenty different
scaffolds tests twenty ideas.

## Step 1 · Generate molecular fingerprints

Convert each molecule, given as a SMILES string, into a **Morgan fingerprint** with RDKit's
`AllChem.GetMorganFingerprintAsBitVect`. That produces a binary vector encoding the structure.

## Step 2 · Cluster with LeaderPicker

Apply the **LeaderPicker** algorithm to those fingerprints to find "leader" molecules, which
act as cluster centroids. The `thresh` parameter sets how similar a molecule must be to a
leader to join its cluster.

## Step 3 · Assign molecules to clusters

Compute the **Tanimoto similarity** between every molecule and each leader, then assign each
molecule to the cluster whose leader it most resembles. `assignPointsToClusters` does this.

## Step 4 · Select representatives and sort

Within each cluster, take a subset (about 1/20th of the cluster). Selection favours molecules
unlike those already picked, which keeps the shortlist diverse. Results are then sorted by
prediction score and cluster id.

In [ ]:
from rdkit.SimDivFilters import rdSimDivPickers
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm import tqdm
from rdkit import Chem

def assignPointsToClusters(picks, fps):
    clusters = defaultdict(list)
    for i, idx in enumerate(picks):
        clusters[i].append(idx)
    sims = np.zeros((len(picks), len(fps)))
    for i in tqdm(range(len(picks))):
        pick = picks[i]
        sims[i, :] = DataStructs.BulkTanimotoSimilarity(fps[pick], fps)
        sims[i, i] = 0  # Don't compare the molecule with itself
    best = np.argmax(sims, axis=0)
    for i, idx in enumerate(best):
        if i not in picks:
            clusters[idx].append(i)
    return clusters

# nominees_filtered holds the compounds that survived the medicinal chemistry filters,
# together with their SMILES and Final_Score.

# Generate Morgan fingerprints using AllChem
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=2048)
fps = [morgan_gen.GetFingerprint(Chem.MolFromSmiles(smi)) for smi in tqdm(nominees_filtered["SMILES"])]

In [ ]:
# Perform clustering using LeaderPicker
lp = rdSimDivPickers.LeaderPicker()
thresh = 0.65  # Minimum distance between cluster centroids
picks = lp.LazyBitVectorPick(fps, len(fps), thresh)  # Using the ExplicitBitVect fingerprints from AllChem
clusters = assignPointsToClusters(picks, fps)

# Assign cluster ids to the prediction_df based on the indices from the clusters
cluster_ids = np.zeros(len(nominees_filtered))  # Initialize cluster_ids for the entire prediction_df

# Make sure to correctly assign the cluster ids
for key, val in clusters.items():
    cluster_ids[val] = key  # Assign the cluster ID to the correct indices

# Add the cluster ids to the prediction_df
nominees_filtered['cluster_id'] = cluster_ids

# Sort the results by Prediction Score and Cluster ID
#nominees_filtered.sort_values(by=["Final_Score", "cluster_id"], ascending=[False, True], inplace=True)

num_clusters = len(set(nominees_filtered["cluster_id"]))
print(f"Number of clusters generated: {num_clusters}")

# Sort the dataframe by Final_Score in descending order
nominees_filtered.sort_values(by=["Final_Score"], ascending=False, inplace=True)

# Select one nominee per cluster: the one with the highest score
best_nominees = nominees_filtered.groupby("cluster_id").first().reset_index()

# Print the selected nominees (one per cluster)
print(best_nominees.head(10))

## The shortlist, as molecules

Numbers are hard to argue with a chemist about; structures are not. These are the
compounds the whole pipeline nominated - one per cluster, so they are structurally
diverse rather than twenty variations on one scaffold.

The table below tracks the known actives through each narrowing step. Expect to lose
some: the drug-likeness filters reject compounds on properties the model never saw,
and taking one representative per cluster deliberately drops actives that sit in a
cluster with a higher-scoring neighbour. Both are reasonable choices, but neither is
free, and it is worth knowing what they cost before you defend the shortlist.

In [ ]:
# Draw the top compounds the pipeline nominated, one per cluster.
top = best_nominees.head(12)
legends = [f"cluster {int(row.cluster_id)}  score {row.Final_Score:.2f}"
           + ("  ACTIVE" if row.LABEL == 1 else "")
           for row in top.itertuples()]

# Track the actives through every narrowing step. Each stage is there for a good
# reason, but each one also costs you hits - and it is worth seeing the price.
print(f"{'stage':<34} {'compounds':>10} {'actives':>8}")
print("-" * 54)
for label, frame in [("nominated by the ensemble", nominees),
                     ("passed medicinal chemistry filters", nominees_filtered),
                     ("one representative per cluster", best_nominees)]:
    print(f"{label:<34} {len(frame):>10} {int(frame['LABEL'].sum()):>8}")

print()
print(f"Showing the top {len(top)} of {len(best_nominees)} cluster representatives.")

plot_molecule_grid(top["SMILES"], legends=legends, n=12, mols_per_row=4)

---

# 📤 Section 9 · Submit Your Nominations

**This is the deliverable, and it is the last thing you do.** Sections 1 to 8 build and
screen; the extra steps above are optional and change nothing here.

Your submission is the **top 200 compounds** from your ranking, saved as a CSV with exactly
two columns:

| Column | What it holds |
|---|---|
| `SMILES` | the compound, as a SMILES string |
| `Prediction_Score` | your model's score for it |

**Rows must be ordered best first**, highest score at the top. The evaluation looks at how
near the top the real actives land, so the order is the answer — not just the set of 200.

Two things to set in the next cell:

- `TEAM_NAME` — the file is written as `<TEAM_NAME>.csv`, so pick something that identifies
  your team and nobody else's.
- `SUBMIT_FROM` — `"single"` uses the Section 8 model, which everyone has. Set it to
  `"ensemble"` only if you ran Extra step 1 and would rather submit that combined ranking.

> **Where it goes.** For now the file lands in the repository's `test/` folder, which
> already holds ten example submissions so the evaluation script has something to read. On
> the day you will be given a different folder — change `SUBMISSION_DIR` and nothing else.

In [ ]:
# ---------------------------------------------------------------------------
# YOUR SUBMISSION - the top 200 compounds from your ranking
# ---------------------------------------------------------------------------
TEAM_NAME = "YourTeamName"          # <-- CHANGE THIS before you run the cell
TOP_N = 200

SUBMIT_FROM = "single"   # "single" = Section 8 model | "ensemble" = Extra step 1
SUBMISSION_DIR = REPO_ROOT / "test"  # <-- on the day, point this at the shared folder
SUBMISSION_DIR.mkdir(exist_ok=True)

if SUBMIT_FROM == "ensemble":
    if "predictions_df_sorted" not in globals():
        raise RuntimeError("Extra step 1 has not been run - set SUBMIT_FROM = 'single'.")
    ranked = (predictions_df_sorted
              .rename(columns={"Final_Score": "Prediction_Score"})
              .loc[:, ["SMILES", "Prediction_Score"]])
else:
    ranked = prediction_df_sorted.loc[:, ["SMILES", "Prediction_Score"]]

submission = ranked.head(TOP_N).reset_index(drop=True)

# Check it before handing it in - a malformed file cannot be scored
assert list(submission.columns) == ["SMILES", "Prediction_Score"], "wrong columns"
assert len(submission) == TOP_N, f"expected {TOP_N} rows, got {len(submission)}"
assert submission["Prediction_Score"].is_monotonic_decreasing, "rows must be best-first"
assert submission["SMILES"].notna().all(), "every row needs a SMILES string"

submission_path = SUBMISSION_DIR / f"{TEAM_NAME}.csv"
submission.to_csv(submission_path, index=False)

print(f"Saved {len(submission)} nominations from the '{SUBMIT_FROM}' ranking")
print(f"  -> {submission_path}")
print()
print(submission.head())

# Only because this particular test set is labelled can we peek at the answer.
# On the day your screening library will be unlabelled and this block will not run.
if "LABEL" in df_test.columns:
    picked = ranked.head(TOP_N).index
    found = int(df_test.loc[picked, "LABEL"].sum())
    total = int(df_test["LABEL"].sum())
    print(f"\nFor reference: your top {TOP_N} contains {found} of the {total} known actives.")

---

# ⏱️ Activity · 5 minutes

Before the hackathon, one short exercise. **You have five minutes.**

So far every model has been LightGBM on ECFP4. That was a choice, not a law. Your job is to
find out how much that choice was worth.

## The task

Compare **two models** across **two fingerprints** — four combinations in all:

| | ECFP4 | TOPTOR |
|---|---|---|
| **LightGBM** | ? | ? |
| **SVM** (`SVC`) | ? | ? |

For each combination, report:

- **AUC-ROC** and **average precision**
- **hit@20** — the metric that matches what a screen does
- **how long it took to train** — this is part of the answer, not a footnote

## How to do it properly

Split `TrainData` into a training part and a validation part, or use cross-validation if you
prefer. **Do not compare them on the test set** — you have already used it, and choosing a
model on it is exactly the mistake the notebook has been warning about.

> **Start the SVMs early.** `SVC` is far slower than LightGBM on data this wide — a single fit takes
> anywhere from half a minute to well over one, against a fraction of a second for
> LightGBM. All four fits together come to roughly two minutes. Kick the cell off and read on while it runs. That wait is
> itself one of the things you are measuring.

## Questions to answer

1. Which fingerprint wins, and by how much? Is the gap big enough to care about?
2. Which model wins? Does it win on every fingerprint, or only one?
3. Look at the training times. Would you accept the slower model for the accuracy it buys?
4. If you had to pick one combination for the hackathon, which — and what would you say if someone asked why?

In [ ]:
# ACTIVITY - write this yourself (5 minutes)
#
# Goal: compare 2 models x 2 fingerprints, and decide which pair you would take
# into the hackathon.
#
#   1. Split TrainData / TrainLabel into a training part and a validation part
#      (train_test_split with stratify=TrainLabel keeps the class balance).
#   2. For each fingerprint in ["ECFP4", "TOPTOR"]:
#          X = process_data(df_train, fingerprint)
#      For each model: LGBMClassifier(...) and SVC(probability=True, ...)
#          fit on the training part, score the validation part.
#   3. Report AUC-ROC, average precision and hit@20 for all four combinations.
#   4. Time each fit - time is part of the answer.
#
# Useful things that already exist in this notebook:
#   process_data(df, column)        build a feature matrix from a fingerprint column
#   hits_at_k(y_true, y_score, ks)  from src.metrics
#   roc_auc_score, average_precision_score
#
# Start here:

import time
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

fingerprints_to_try = ["ECFP4", "TOPTOR"]

print("Your turn - four combinations to compare.")
print("Remember: the SVM fits take about a minute each, so start them running.")

---

# 🏆 Hackathon Details

You already have everything you need. The notebook above is a complete, working pipeline —
the hackathon is about making it *yours*.

Nobody is expected to invent something from scratch in an hour. The teams that do well are
usually the ones that change **one thing** thoughtfully, measure whether it actually helped,
and keep it only if it did.

> Anything written as `_____` below will be filled in on the day.

## The format

| | |
|---|---|
| Time | **one hour** |
| Team size | `_____` |
| Submissions allowed | `_____` |
| Leaderboard | updates every **5 minutes**, at `_____` |
| Mentors | on the floor throughout — please use them |

Mentors are there to unstick you, not to judge you. If something errors, if a number looks
impossible, or if you just want a second opinion on an idea, ask early. An hour disappears
quickly, and ten minutes lost to a stack trace is ten minutes not spent on the chemistry.

## The data

| Role | File | Compounds | Labels | SMILES | What it is for |
|---|---|---|---|---|---|
| **Train** | `_____` | `_____` | yes | `_____` | fit your model and validate it |
| **Test** | `_____` | `_____` | hidden | yes | the compounds you score and rank |

**Where to find them:** `_____`

In the practice notebook these were `data/sample-train.parquet` (4,000 compounds, balanced
50/50) and `data/sample-test.parquet` (5,000 compounds, 9 actives). The real files are
larger, and the test labels are hidden from you — that is what the leaderboard is for.

**Known actives in the test set:** `_____`

## What you submit

A CSV named after your team, in exactly the format Section 9 writes:

| Column | Contents |
|---|---|
| `SMILES` | the compound |
| `Prediction_Score` | your model's score for it |

- **200 rows**, ordered best first
- Saved to `_____`
- Set `TEAM_NAME` and `SUBMISSION_DIR` at the top of the Section 9 cell

Section 9 checks the file before it writes anything. If an assertion fails, the file is
wrong — fix it rather than submitting anyway.

## How you are scored

**Metric:** `_____`

Whatever the exact formula, it rewards putting real actives near the top of your 200.
Accuracy is not used and would be meaningless here: with actives this rare, a model that
nominates nothing at all scores almost perfectly. The Metrics section earlier in this
notebook shows that happening, if it still feels surprising.

> **Optimise for the metric you are scored on, not the one that looks best.** A validation
> AUC that improves without moving your hit count is not progress.

## Ideas worth trying in an hour

Roughly in order of effort:

1. **Change the fingerprint.** `selected_fps` in Section 3 — one word, and ECFP4 is not always the winner.
2. **Use the ensemble.** Extra step 1 already beat the single model at the sharp end. Set `SUBMIT_FROM = "ensemble"` in Section 9.
3. **Fuse fingerprints.** The optional cell in Section 3 builds them for train and test. Swap both, or neither.
4. **Tune harder.** Section 6 tries ten configurations by hand. Widen the grid, or hand it to `RandomizedSearchCV`.
5. **Change the class balance.** The optional cell in Section 3 varies negatives per positive. Real screening data holds far more inactives than the practice set does.
6. **Try a different model.** Section 4 defines four alternatives, each a one-line swap.
7. **Split by group.** The discussion after Section 5 argues that a random split flatters a DEL model. A group-aware split will *lower* your validation score and may well *raise* your leaderboard score.

> **Worth heeding.** Every one of those can be judged *before* you submit, with
> cross-validation on the training data. Submitting blind and reading the leaderboard is a
> slow way to learn — and with a limited number of submissions, an expensive one.

## Before you submit

- ☐ `TEAM_NAME` set to your team, not `YourTeamName`
- ☐ `SUBMISSION_DIR` pointing at the shared folder
- ☐ Section 9 ran with no assertion error
- ☐ 200 rows, two columns, ranked best first
- ☐ You can say in one sentence what you changed and why

Good luck — the pipeline already works. Go and make it better.